# Model 3: Gradient Boosting

This notebook builds and tunes HistGradientBoosting models.

In [1]:
from pathlib import Path
import json
import sys

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_split
from src.evaluate import cv_report, save_result
from src.pipeline import build

REPORTS_DIR = PROJECT_ROOT / 'reports'
BEST_PARAMS_PATH = REPORTS_DIR / 'best_params.json'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Split first

In [2]:
X_train, X_test, y_train, y_test = get_split()
print(f'Training rows: {len(X_train):,}')
print(f'Held-out test rows: {len(X_test):,}')
print(f'Training yes-rate: {y_train.mean():.3%}')

Training rows: 36,168
Held-out test rows: 9,043
Training yes-rate: 11.698%


## HistGradientBoosting baseline and tuning

In [3]:
# HistGradientBoosting is sklearn's fast implementation of the GBM idea from lectures.
hgb_baseline = build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42))
baseline_row = cv_report('HistGradientBoosting', hgb_baseline, X_train, y_train)
save_result(baseline_row, REPORTS_DIR / 'model_comparison.csv')
baseline_row

{'name': 'HistGradientBoosting',
 'pr_auc_mean': np.float64(0.44018308503680104),
 'pr_auc_std': np.float64(0.00953980096325405),
 'roc_auc_mean': np.float64(0.7951364280236309),
 'roc_auc_std': np.float64(0.0027595737870123958),
 'f1_mean': np.float64(0.44586406900565534),
 'f1_std': np.float64(0.009917654521236146),
 'precision_mean': np.float64(0.3507823608010944),
 'precision_std': np.float64(0.01201596383975227),
 'recall_mean': np.float64(0.6121513560585128),
 'recall_std': np.float64(0.006734125666640066),
 'train_pr_auc_mean': np.float64(0.5224076890492517),
 'train_pr_auc_std': np.float64(0.009359372703182727),
 'fit_time_mean': np.float64(2.392582082748413)}

In [4]:
hgb_search = RandomizedSearchCV(
    estimator=build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42)),
    param_distributions={
        'model__learning_rate': [0.02, 0.05, 0.1, 0.2],
        'model__max_depth': [None, 3, 5, 7],
        'model__max_leaf_nodes': [15, 31, 63, 127],
        'model__min_samples_leaf': [10, 20, 50, 100],
        'model__l2_regularization': [0.0, 0.1, 1.0, 10.0],
    },
    n_iter=25,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
hgb_search.fit(X_train, y_train)
hgb_tuned_row = cv_report('HistGradientBoosting-tuned', hgb_search.best_estimator_, X_train, y_train)
save_result(hgb_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(hgb_search.best_params_)
hgb_tuned_row

{'model__min_samples_leaf': 50, 'model__max_leaf_nodes': 31, 'model__max_depth': None, 'model__learning_rate': 0.1, 'model__l2_regularization': 10.0}


{'name': 'HistGradientBoosting-tuned',
 'pr_auc_mean': np.float64(0.4489031385914375),
 'pr_auc_std': np.float64(0.011179245171381661),
 'roc_auc_mean': np.float64(0.7977679267814354),
 'roc_auc_std': np.float64(0.0008955211949887608),
 'f1_mean': np.float64(0.44629131359350793),
 'f1_std': np.float64(0.0072685904741842935),
 'precision_mean': np.float64(0.34740152771926525),
 'precision_std': np.float64(0.01008253642039383),
 'recall_mean': np.float64(0.6244394762770005),
 'recall_std': np.float64(0.007022697313936853),
 'train_pr_auc_mean': np.float64(0.5195690127512929),
 'train_pr_auc_std': np.float64(0.006918019124769132),
 'fit_time_mean': np.float64(4.936135721206665)}

## Preserve the best-parameter record

In [ ]:

if BEST_PARAMS_PATH.exists():
    best_params = json.loads(BEST_PARAMS_PATH.read_text())
else:
    best_params = {}
best_params['HistGradientBoosting-tuned'] = hgb_search.best_params_
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2) + '\n')
print(json.dumps(best_params, indent=2))
